In [ ]:
# Make sure you are in your deep learning environment

import os
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import wandb
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# ==========================================
# PHASE 1: Data Acquisition
# ==========================================
print("📥 Loading 200k production dataset from Hugging Face...")
hf_dataset = load_dataset("lanretto/discourse_quality", data_files="full_labeled_flattened.csv", split="train")
df = hf_dataset.to_pandas()

target_cols = [
    'level_of_justification', 'respect_towards_demands', 'respect_towards_counterarguments',
    'content_of_justification', 'respect_towards_groups', 'constructive_politics'
]
df_model = df[['comment'] + target_cols].dropna()
df_model['comment'] = df_model['comment'].astype(str)

print("🔄 'Zipping' dimensions and splitting data...")
df_model['labels'] = df_model[target_cols].values.tolist()
train_df, test_df = train_test_split(df_model, test_size=0.2, random_state=42)

train_dataset = Dataset.from_pandas(train_df[['comment', 'labels']], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[['comment', 'labels']], preserve_index=False)

# ==========================================
# PHASE 2: Tokenization (Swapped to BERT-large!)
# ==========================================
print("⚙️ Loading the bert-large-uncased Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("bert-large-uncased")

def tokenize_function(examples):
    return tokenizer(examples['comment'], padding="max_length", truncation=True, max_length=128)

print("Tokenizing data...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# ==========================================
# PHASE 3: Model Architecture (Swapped to BERT-large!)
# ==========================================
print("🧠 Loading the BERT-large Master Model...")
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-large-uncased",
    num_labels=6,
    problem_type="regression"
)

# ==========================================
# PHASE 4: Validation Grader
# ==========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    metrics = {}
    total_mse = 0
    for i, col_name in enumerate(target_cols):
        mse = mean_squared_error(labels[:, i], logits[:, i])
        metrics[f"mse_{col_name}"] = mse
        total_mse += mse
    metrics["mse_OVERALL"] = total_mse / 6
    return metrics

# ==========================================
# PHASE 5: W&B Integration & Training
# ==========================================
print("🔗 Connecting to Weights & Biases...")
os.environ["WANDB_API_KEY"] = "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"

wandb.init(
    entity="niyimoluga85-universty-of-technology-graz",
    project="DQI_Multi_Regression",
    name="bert_large_6_dim", # Updated tracking name
)

training_args = TrainingArguments(
    output_dir="./results_bert_large_master",
    eval_strategy="steps",
    logging_strategy="steps",
    save_strategy="steps",
    eval_steps=200,
    logging_steps=200,
    save_steps=200,
    save_total_limit=2,
    learning_rate=2e-5,
    # THE MEMORY TRICK HAPPENS HERE:
    per_device_train_batch_size=8, 
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="mse_OVERALL",
    greater_is_better=False,
    report_to="wandb"
)

class PureMSETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.MSELoss(reduction="none")
        per_dim_errors = loss_fct(logits, labels)
        mse_per_dim = per_dim_errors.mean(dim=0)
        master_mse = mse_per_dim.mean()
        
        if self.state.global_step % self.args.logging_steps == 0 and self.args.should_log:
            log_dict = {}
            for i, col_name in enumerate(target_cols):
                log_dict[f"train/mse_{col_name}"] = mse_per_dim[i].item()
            log_dict["train/mse_OVERALL"] = master_mse.item()
            self.log(log_dict)
            
        return (master_mse, outputs) if return_outputs else master_mse

trainer = PureMSETrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("\n🚀 Starting BERT-large Multi-Output Training...")
trainer.train()
wandb.finish()
print("\n✅ Training Complete!")

/home/larry/miniconda3/envs/discourse_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📥 Loading 200k production dataset from Hugging Face...
🔄 'Zipping' dimensions and splitting data...
⚙️ Loading the bert-large-uncased Tokenizer...
Tokenizing data...


Map: 100%|██████████| 41170/41170 [00:03<00:00, 12697.88 examples/s]


🧠 Loading the BERT-large Master Model...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-large-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


...


🚀 Starting BERT-large Multi-Output Training...


Step,Training Loss,Validation Loss,Mse Level Of Justification,Mse Respect Towards Demands,Mse Respect Towards Counterarguments,Mse Content Of Justification,Mse Respect Towards Groups,Mse Constructive Politics,Mse Overall
200,0.076100,0.060216,0.054911,0.069850,0.065639,0.072793,0.040541,0.057565,0.060216
400,0.059500,0.055177,0.051383,0.062129,0.060267,0.068330,0.037350,0.051603,0.055177
600,0.057100,0.053561,0.049787,0.060908,0.058809,0.066207,0.035822,0.049831,0.053561
800,0.053100,0.052357,0.048753,0.058387,0.057670,0.064812,0.035397,0.049124,0.052357


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7d2d14bb54f0>> (for post_run_cell), with arguments args (<ExecutionResult object at 7d2e8061abd0, execution_count=1 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7d2e8061aff0, raw_cell="# Make sure you are in your deep learning environm.." transformed_cell="# Make sure you are in your deep learning environm.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Bartemis/home/larry/BERT-large.ipynb#W0sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost